In [ ]:
pip install ugot

In [ ]:
pip install opencv-python

In [ ]:
pip install matplotlib

In [2]:
from ugot import ugot
got = ugot.UGOT()
got.initialize('192.168.1.13')
print("UGOT connected!")

192.168.1.13:50051
UGOT connected!


In [ ]:
# import YOLO
import ultralytics
from ultralytics import YOLO
import cv2
from IPython.display import Image, display

ultralytics.checks()
model = YOLO("yolo26n.pt")
print("Model is ready.")

## Level 5 Term 1 sesssion 4

1. bounding boxes in video

In [ ]:
# bounding boxes on an uploaded video file
import os
import time
from collections import Counter

sample_video = "high_way_video.mp4"                   # change this to your video file name
output_video = "high_way_video_boxes.mp4"       # the new video with bounding boxes

if not os.path.exists(sample_video):
    print(f"Cannot find {sample_video}. Put the video in the same folder as this notebook.")
else:
    video = cv2.VideoCapture(sample_video)

    # Step 0: read the information of the video
    fps = video.get(cv2.CAP_PROP_FPS)                        # frames per second
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"The video has {total_frames} frames, {fps:.0f} frames per second, size {width}x{height}.")
    print(f"The video is about {total_frames/fps:.1f} seconds long.")

    # the "recorder" that saves the frames with boxes into a new video file
    writer = cv2.VideoWriter(output_video,cv2.VideoWriter_fourcc(*"mp4v"),fps,(width,height))

    handle = None
    frame_number = 0
    start_time = time.time()

    while True:
        # Step 1: read one frame, stop when the video ends
        frame_ok,frame = video.read()
        if not frame_ok:
            break
        frame_number = frame_number + 1

        # Step 2: let YOLO find the objects in this frame
        results = model(frame,verbose = False)

        # Step 3: draw a box, the name and the confidence for each object
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = model.names[class_id]
            x1,y1,x2,y2 = map(int,box.xyxy[0])
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame,f"{name} {conf:.2f}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

        # Step 4: save the frame into the new video
        writer.write(frame)

        # Step 5: show the frame in the same display area
        jpeg_ok,jpeg = cv2.imencode(".jpg",frame)
        if jpeg_ok:
            if handle is None: # create the display area for the first time
                handle = display(Image(data=jpeg.tobytes()),display_id = True)
            else: # update the same display area
                handle.update(Image(data=jpeg.tobytes()))

    video.release()
    writer.release()
    print(f"Finished {frame_number} frames in {time.time()-start_time:.1f} seconds.")

2. take video using laptop camera

In [ ]:
import time

# open the laptop camera
camera = cv2.VideoCapture(0,cv2.CAP_AVFOUNDATION) # DSHOW
time.sleep(1)
print("Camera is ready.")

if not camera.isOpened():
    print("The camera is not working.")
else:
    print("The camera is on.")
    time.sleep(1)

    handle = None
    start_time = time.time()

    while time.time() - start_time<5:
        frame_ok,frame = camera.read()
        if frame_ok:
            jpeg_ok,jpeg = cv2.imencode(".jpg",frame)
            if jpeg_ok:
                if handle is None: # create the display area for the first frame
                    handle =display(Image(data=jpeg.tobytes()),display_id=True)
                else: # update the same display area
                    handle.update(Image(data=jpeg.tobytes()))
        time.sleep(0.02)

    camera.release()

3. laptop camera with YOLO

In [ ]:
import time

# open the laptop camera
camera = cv2.VideoCapture(0,cv2.CAP_AVFOUNDATION) 
time.sleep(1)
print("Camera is ready.")

if not camera.isOpened():
    print("The camera is not working.")
else:
    print("The camera is on.")
    time.sleep(1)

    handle = None
    start_time = time.time()

    while time.time() - start_time<10:
        frame_ok,frame = camera.read()
        if not frame_ok:
            continue
        
        # YOLO block
        results = model(frame,verbose = False)
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = model.names[class_id]
            x1,y1,x2,y2 = map(int,box.xyxy[0])
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame,f"{name} {conf:.2f}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
        
        # handle block
        jpeg_ok,jpeg = cv2.imencode(".jpg",frame)
        if jpeg_ok:
            if handle is None: # create the display area for the first frame
                handle =display(Image(data=jpeg.tobytes()),display_id=True)
            else: # update the same display area
                handle.update(Image(data=jpeg.tobytes()))

    camera.release()